# GOES-East ABI Cloud Animation Pipeline

This notebook is a starter workflow for downloading, processing, and animating imagery from the **GOES-East Advanced Baseline Imager (ABI)** over a configurable region. You set a date, a time window, a geographic extent, and a frame cadence, and the pipeline pulls the matching ABI scans, renders each as a map frame, and assembles them into an animation.

The pipeline is deliberately general. The same machinery animates any mesoscale cloud phenomenon — convective initiation, lake-effect snow bands, fog burn-off, frontal passages, outflow boundaries — because nothing in the download or animation loop is specific to one weather pattern. What you study is determined by **which case day and region you choose**, not by the tooling. The configuration block below exposes the knobs (satellite, product, scan domain, band, extent, cadence) so you can retarget the pipeline without rewriting it.

## The GOES-East satellites

NOAA's Geostationary Operational Environmental Satellites provide the imagery:

- **GOES-16** — GOES-East from 2017-12-18 to 2025-04-07
- **GOES-19** — operational GOES-East from 2025-04-04 (UTC)

The two are the same GOES-R-series hardware with equivalent ABI imagers, so band products and file structures are interchangeable; only the satellite number in the configuration changes with the date.

The default path uses **ABI visible-band imagery**, which shows the actual cloud field directly and avoids the memory-heavy HRRR native-level cloud-water/cloud-ice fields. HRRR low cloud cover and 10 m winds are left as an optional second stage.

## Example application: lake breezes

One thing this pipeline is well-suited to find is a **lake breeze**. On a warm, mostly sunny day, cool stable air flowing inland off Lake Michigan suppresses cloud formation over and near the water while clouds build over the heated land — producing a sharp, curved **lake-breeze front** that marks the inland edge of the marine air. Animating the visible cloud field lets you watch that boundary form mid-morning, penetrate inland through the afternoon, and collapse toward evening.

Lake breezes make a good worked example precisely because the signature is *visual and mesoscale*: a moving cloud-edge boundary is exactly what a frame-by-frame animation reveals and a single still does not. To use the pipeline this way, pick a warm-season day with weak synoptic wind and start with daylight hours. (Confirming a cloud boundary as a true lake-breeze front ultimately wants surface wind data — the optional HRRR stage at the end — but the cloud animation is how you spot candidate days.)

## References

GOES-2-Go documentation:
- https://goes2go.readthedocs.io/en/latest/user_guide/index.html
- https://goes2go.readthedocs.io/en/latest/reference_guide/index.html

## Install notes

Recommended conda-forge packages:

```bash
mamba install -c conda-forge goes2go cartopy matplotlib imageio pillow xarray netcdf4 s3fs
```

Optional HRRR overlay later:

```bash
mamba install -c conda-forge herbie-data cfgrib eccodes
```

The default notebook uses GOES visible-band imagery. That is lighter than GOES true-color RGB and much lighter than the failed HRRR native-level cloud notebook.


In [ ]:
from pathlib import Path

import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from goes2go import GOES

In [ ]:
#from datetime import datetime, timedelta

## Case settings

The date below is just a placeholder. For a real lake-breeze case, choose a mostly sunny warm-season day and start with daylight hours.

Times are UTC. Chicago is UTC-5 during Central Daylight Time, so:

```text
14 UTC = 9 AM CDT
23 UTC = 6 PM CDT
```


In [ ]:
# Placeholder case date. Change this later.
CASE_DATE = "2025-05-07"

START_UTC = f"{CASE_DATE} 14:00"
END_UTC = f"{CASE_DATE} 23:00"

# Frame cadence. Coarser = fewer downloads (gentler first run); finer = smoother
# animation. CONUS scans every 5 min, so "5min" is the practical floor for DOMAIN="C".
# A 9-hour window at "30min" = 19 frames; at "5min" = 108 frames.
FRAME_FREQ = "30min"
#FRAME_FREQ = "5min"

step = pd.Timedelta(FRAME_FREQ).to_timedelta64()
frame_times = list(np.arange(np.datetime64(START_UTC),
                             np.datetime64(END_UTC) + np.timedelta64(1, "m"),
                             step))

# Southern Lake Michigan / Chicago extent:
# [west, east, south, north]
EXTENT = [-88.6, -85.8, 40.8, 43.3]

CHICAGO_LON = -87.6298
CHICAGO_LAT = 41.8781

DATA_DIR = Path("data/goes")
FRAME_DIR = Path("frames/goes")
OUT_DIR = Path("animations")

DATA_DIR.mkdir(parents=True, exist_ok=True)
FRAME_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)


## GOES object

This uses **GOES-East ABI Cloud and Moisture Imagery, CONUS domain**.

Default is **band 2**, the red visible band. It is good for daytime cloud animations and is much smaller than downloading a full 16-channel RGB product for every frame.


In [ ]:
# --- GOES product configuration ---
# Each parameter below has alternatives (commented). Swap them in to study
# different phenomena, regions, or times of day.

SATELLITE = 19          # GOES-East. Use 16 for dates before 2025-04-04.
# SATELLITE = 18        # GOES-West (Pacific, western CONUS)

PRODUCT = "ABI-L2-CMIP" # Single-band Cloud & Moisture Imagery.
# PRODUCT = "ABI-L2-MCMIPC"  # All 16 bands in one file — needed for RGB composites.

DOMAIN = "C"            # CONUS, 5-min cadence.
# DOMAIN = "F"          # Full Disk, 10-min, whole hemisphere (large files).
# DOMAIN = "M1"         # Mesoscale sector 1, 30-60 SEC cadence, small area.
# DOMAIN = "M2"         # Mesoscale sector 2, 30-60 sec cadence.

BAND = 2                # Red visible, 0.5 km — best daytime cloud detail.
# BAND = 1              # Blue visible (0.5 µm)
# BAND = 3              # "Veggie" near-IR (0.86 µm)
# BAND = 13             # Clean IR window (10.3 µm) — works at NIGHT, when
                        #   visible bands are dark. Use for after-sunset cases.

In [ ]:
G = GOES(
    satellite=SATELLITE,   # 16 before April 2025 19 after
    product=PRODUCT,
    domain=DOMAIN,
    bands=BAND,
)

# List available files for the case window.
files = G.df(start=START_UTC, end=END_UTC, ignore_missing=True)
files.head()

In [ ]:
print(f"Found {len(files)} GOES files between {START_UTC} and {END_UTC}.")
files.tail()

## Download/read one test frame

This cell downloads and opens the GOES file nearest the requested time.

If this works, the animation loop below is much more likely to work.


In [ ]:
TEST_TIME = f"{CASE_DATE} 18:00"

ds = G.nearesttime(
    TEST_TIME,            
    within="30min",  
    save_dir=DATA_DIR,
)

ds

## Plotting function

This function plots one GOES visible-band frame over southern Lake Michigan.

The function assumes the GOES-2-Go `FOV` accessor is available. It supplies the correct geostationary projection and `imshow` arguments for the ABI grid.


In [ ]:
def plot_goes_band(ds, ax=None, title=None):
    """Plot one GOES ABI frame over the configured extent.

    Defaults (cmap='gray', vmin=0, vmax=1) are tuned for VISIBLE reflectance
    bands (1-6). For an IR band such as 13, the values are brightness
    temperatures (~180-330 K) and you'd want cmap='gray_r', vmin~190, vmax~300.
    """
    if ax is None:
        fig, ax = plt.subplots(
            figsize=(8, 7),
            subplot_kw={"projection": ds.FOV.crs},
            constrained_layout=True,
        )
    else:
        fig = ax.figure

    # CMI = Cloud and Moisture Imagery (reflectance factor for visible bands).
    im = ax.imshow(
        ds.CMI,
        **ds.FOV.imshow_kwargs,
        cmap="gray",
        vmin=0,
        vmax=1,
    )

    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor="none", edgecolor="black", linewidth=0.5)
    ax.add_feature(cfeature.LAKES, facecolor="none", edgecolor="black", linewidth=0.8)
    ax.add_feature(cfeature.STATES, edgecolor="0.4", linewidth=0.5)

    ax.plot(
        CHICAGO_LON,
        CHICAGO_LAT,
        marker="o",
        markersize=4,
        color="red",
        transform=ccrs.PlateCarree(),
        label="Chicago",
    )

    if title is None:
        try:
            scan_start = np.datetime_as_string(ds.t.values, unit="m")
        except Exception:
            scan_start = ""
        title = f"GOES-East ABI band {BAND}\n{scan_start} UTC"

    ax.set_title(title)
    ax.legend(loc="lower left")

    return fig, ax, im

In [ ]:
fig, ax, im = plot_goes_visible(ds)
plt.show()

## Build a short animation

This first version uses hourly frames. After the workflow is stable, reduce `FRAME_FREQ` to `"30min"` or `"15min"`.

The loop saves PNG frames first, then assembles a GIF. Saving frames makes debugging much easier than trying to animate directly in memory.


In [ ]:
# Clear old frames from a previous run so the GIF isn't built from stale images.
for old_png in FRAME_DIR.glob(f"goes_band{BAND}_*.png"):
    old_png.unlink()

saved_frames = []

for i, t in enumerate(frame_times):
    t_string = str(t).replace("T", " ")
    print(f"Frame {i + 1}/{len(frame_times)}: {t_string}")

    try:
        ds_i = G.nearesttime(t_string, within="30min", save_dir=DATA_DIR)
    except Exception as e:
        print(f"  skipped (no/failed file): {e}")
        continue

    fig, ax, im = plot_goes_band(
        ds_i,
        title=f"GOES-East ABI band {BAND}\n{t_string} UTC",
    )

    frame_path = FRAME_DIR / f"goes_band{BAND}_{i:03d}.png"
    fig.savefig(frame_path, dpi=130)
    plt.close(fig)

    saved_frames.append(frame_path)
    ds_i.close()

print(f"Saved {len(saved_frames)} frames in {FRAME_DIR}")

In [ ]:
gif_path = OUT_DIR / f"goes_band{BAND}_{CASE_DATE}.gif"

images = [imageio.imread(frame) for frame in saved_frames]
imageio.mimsave(gif_path, images, duration=0.25)

gif_path

## Display the GIF in the notebook

If this does not display in VSCode, open the GIF file directly from the `animations/` folder.


In [ ]:
from IPython.display import Image, display

display(Image(filename=str(gif_path)))

# Optional: HRRR overlay later

Once the GOES-only animation works, the next step is to overlay HRRR **surface** fields:

- `UGRD` and `VGRD` at 10 m for wind vectors
- `LCDC` for low cloud cover, if desired

Avoid native HRRR cloud water/ice fields such as `CLMR` and `CIMIXR` until the workflow is stable. Those are vertical model-level fields and can overwhelm laptop memory.


In [ ]:
# Optional diagnostic only. Do not run until Herbie is installed and GOES-only animation works.

# from herbie import Herbie
#
# H = Herbie(
#     f"{CASE_DATE} 12:00",
#     model="hrrr",
#     product="sfc",
#     source="aws",
#     fxx=6,
# )
#
# inv = H.inventory()
# inv[inv.search_this.str.contains("UGRD:10 m|VGRD:10 m|LCDC", case=False, na=False)]

## Notes for later improvements

Possible next improvements:

1. Use 10- or 15-minute GOES frames after the hourly workflow works.
2. Add HRRR 10 m wind arrows every hour.
3. Add a lake-breeze case-day list and loop over multiple days.
4. Save MP4 instead of GIF for smoother animations.
5. Try GOES true-color or natural-color RGB after the visible-band version works.
